In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal"))
from multimodalpy.get_network import main


NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = Path.home() / "Desktop/descargas_multimodalpy/output_Valencia"
boundaries = Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/data/boundaries/shapes/capa_Valencia.shp"

AREA_NAME = "Valencia"
CRS = "epsg:25830"
MODES = ["driving", "walking", "bike", "bus", "train"]


In [2]:
from network_qa import (
    run_all_checks_osm,
    results_to_frame,
    style_results,
    summary_matrix,
    style_summary_matrix,
    show_issues,
)


In [3]:
STEM_BY_MODE = {
    "driving": "osm_driving",
    "walking": "osm_walking",
    "bike": "osm_bike",
    "train": "gtfs_20260902_080005_metro_valencia",
    "bus": "gtfs_20260909_020020_genvalenciana_interurbano",
}


def load_network(mode: str, fmt: str, gpkg_path=None):
    """Devuelve (nodes, edges) para `mode` en el formato `fmt`.

    fmt: "geojson" | "shapefile" | "geopackage"

    `output_folder`/`gpkg_path` se convierten a Path aqui mismo, para que
    esta funcion funcione sin importar si en la celda de configuracion los
    definiste como str o como Path (el operador "/" solo existe entre
    objetos Path, y ese fue el TypeError si veniamos de un str).
    """
    stem = STEM_BY_MODE[mode]
    folder = Path(output_folder)

    if fmt == "geojson":
        nodes = gpd.read_file(folder / f"{stem}_nodes.geojson")
        edges = gpd.read_file(folder / f"{stem}_edges.geojson")
    elif fmt == "shapefile":
        nodes = gpd.read_file(folder / f"{stem}_nodes.shp")
        edges = gpd.read_file(folder / f"{stem}_edges.shp")
    elif fmt == "geopackage":
        if gpkg_path is None:
            raise ValueError("fmt='geopackage' requiere gpkg_path=...")
        gpkg_path = Path(gpkg_path)
        nodes = gpd.read_file(gpkg_path, layer=f"{stem}_nodes")
        edges = gpd.read_file(gpkg_path, layer=f"{stem}_edges")
    else:
        raise ValueError(f"Formato desconocido: {fmt!r}")

    return nodes, edges


def run_checks_for_all_modes(fmt: str, gpkg_path=None) -> dict:
    """Corre run_all_checks_osm para cada modo en MODES, en el formato dado.

    Devuelve {modo: [CheckResult, ...]} — usa `show_results()` (celda de
    abajo) para verlo como tabla en vez de leer el diccionario a mano.
    """
    results_by_mode = {}
    for mode in MODES:
        nodes, edges = load_network(mode, fmt, gpkg_path=gpkg_path)
        # verbose=False: nada de log linea-por-linea aqui, solo guardamos
        # los resultados. El resumen se revisa en la siguiente celda como
        # tabla coloreada, mucho mas facil de escanear que el print crudo.
        results_by_mode[mode] = run_all_checks_osm(nodes, edges, verbose=False)
    return results_by_mode


In [ ]:
def show_results(results_by_mode: dict):
    """Muestra la matriz coloreada check x modo para un formato."""
    matrix = summary_matrix(results_by_mode)
    return style_summary_matrix(matrix)


## GeoJSON

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geojson",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

In [4]:
results_geojson = run_checks_for_all_modes("geojson")
show_results(results_geojson)

=== SUMMARY ===
26 passed, 1 failed, 0 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)


DataSourceError: /Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia/gtfs_20260909_020020_genvalenciana_interurbano_nodes.geojson: No such file or directory

Ejemplo de cómo abrir el detalle de un check que salió en rojo arriba (ajusta el nombre del check y el modo a lo que veas en tu matriz):

```python
show_issues(results_geojson["bus"], "trip_count == sum(hourly_trip_counts)")
```

## Shapefile

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="shapefile",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

results_shapefile = run_checks_for_all_modes("shapefile")
show_results(results_shapefile)


## GeoPackage

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geopackage",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

gpkg_path = output_folder / "valencia.gpkg"
results_geopackage = run_checks_for_all_modes("geopackage", gpkg_path=gpkg_path)
show_results(results_geopackage)
